In [12]:
import numpy as np
import bz2

# Load gene sequences from .bz2 file
def extract_gene_sequences(filepath):
    gene_sequences = {}
    try:
        with bz2.open(filepath, 'rt') as file:
            for line in file:
                parts = line.strip().split(' \\ ')
                if len(parts) == 2:
                    gene, dna = parts[0].strip(), parts[1].strip()
                    gene_sequences[gene] = dna
                else:
                    print(f"Warning: malformed line skipped -> {line.strip()}")
    except Exception as err:
        print(f"Failed to read sequence file: {err}")
    print(f"Total genes parsed: {len(gene_sequences)}")
    return gene_sequences

# Parse the base counts from the given matrix file
def read_counts_data(matrix_file):
    bases = ['a', 'c', 'g', 't']
    matrix = {base: [] for base in bases}
    with open(matrix_file) as file:
        for line in file:
            line = line.strip().lower()
            if not line or line[0] not in bases:
                continue
            parts = line.replace('|', '').split()
            base = parts[0]
            values = list(map(int, parts[1:]))
            matrix[base].extend(values)
    return np.array([matrix[base] for base in bases])

# Generate log-odds weight matrix
def build_weight_matrix(count_data):
    pseudo = 1
    adjusted_counts = count_data + pseudo
    column_sums = adjusted_counts.sum(axis=0)
    freq_matrix = adjusted_counts / column_sums
    background = 0.25
    weight_matrix = np.log2(freq_matrix / background)
    return weight_matrix

# Generate adjusted frequency matrix F'
def adjusted_frequency_matrix(count_data):
    pseudo = 1
    total_per_col = count_data.sum(axis=0) + 4 * pseudo
    return (count_data + pseudo) / total_per_col

# Slide motif across sequences and calculate top matches
def find_top_motif_sites(weights, dna_dict, top_k=30):
    base_index = {'a': 0, 'c': 1, 'g': 2, 't': 3}
    motif_width = weights.shape[1]
    scored_sites = []

    for gene, dna_seq in dna_dict.items():
        dna_seq = dna_seq.lower()
        if len(dna_seq) < motif_width:
            print(f"Skipping {gene}, sequence too short.")
            continue
        for i in range(len(dna_seq) - motif_width + 1):
            motif = dna_seq[i:i + motif_width]
            try:
                score = sum(weights[base_index[b], j] for j, b in enumerate(motif) if b in base_index)
                scored_sites.append((gene, score, motif, i))
            except KeyError:
                continue

    scored_sites.sort(key=lambda x: x[1], reverse=True)
    return scored_sites[:top_k]

# File paths
sequence_file = 'E_coli_K12_MG1655.400_50.bz2'
matrix_file = 'argR-counts-matrix.txt'

# Run everything
gene_data = extract_gene_sequences(sequence_file)
if not gene_data:
    print("Error: No sequences loaded.")
else:
    raw_counts = read_counts_data(matrix_file)
    pwm_matrix = build_weight_matrix(raw_counts)
    freq_adjusted = adjusted_frequency_matrix(raw_counts)

    print("\nPosition Weight Matrix (PWM):")
    print(pwm_matrix)
    print("\nAdjusted Frequency Matrix F'(b, j):")
    print(freq_adjusted)

    top_matches = find_top_motif_sites(pwm_matrix, gene_data)
    print(f"\nTop {len(top_matches)} High-Scoring Motif Matches:")
    for gene, score, motif, index in top_matches:
        print(f"{gene} | Score: {score:.2f} | Start: {index} | Match: {motif}")


Total genes parsed: 4319

Position Weight Matrix (PWM):
[[ 0.21572869  0.74624341  1.50523531  0.36773178 -0.63226822 -1.36923381
   1.50523531  1.50523531 -0.95419631  0.50523531  0.21572869 -0.36923381
   0.04580369  1.74624341 -0.63226822 -1.36923381 -1.36923381  1.74624341]
 [ 0.04580369 -0.63226822 -1.95419631 -0.14684139 -1.36923381 -0.95419631
  -0.95419631 -0.95419631 -1.95419631 -2.95419631 -1.36923381 -2.95419631
   0.04580369 -2.95419631 -0.95419631 -0.95419631  1.68965988 -2.95419631]
 [-0.95419631 -1.36923381 -1.95419631  0.21572869 -1.36923381  1.50523531
  -1.36923381 -1.36923381 -2.95419631 -1.95419631 -2.95419631 -1.95419631
  -2.95419631 -1.95419631 -2.95419631  1.04580369 -2.95419631 -1.36923381]
 [ 0.36773178  0.36773178 -0.63226822 -0.63226822  1.36773178 -1.95419631
  -1.95419631 -1.95419631  1.63076619  1.13326653  1.21572869  1.50523531
   0.85315861 -1.95419631  1.43812111  0.04580369 -1.95419631 -2.95419631]]

Adjusted Frequency Matrix F'(b, j):
[[0.29032258 0